# Pretrained Siamese Training

In [ ]:
import copy
from pathlib import Path
import torch
from torch import nn
from utils import set_seed, get_signature_paths, split_writers, sample_triplets, run_epoch, plot_history

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_DIR = Path("model")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
BATCH_SIZE = 16
EPOCHS = 10
MARGIN = 0.2
FORCE_RETRAIN = False

set_seed(SEED)
writers = get_signature_paths("dataset/process")
train_w, val_w, test_w = split_writers(writers, SEED)
print("Device:", DEVICE)
print("Train:", len(train_w), "Validation:", len(val_w), "Test:", len(test_w))

In [ ]:
MODEL_NAMES = ["resnet18", "mobilenet_v3_small", "efficientnet_b0", "densenet121"]
IMAGE_SIZE = (224, 224)
EMBEDDING_DIM = 128
FREEZE_EPOCHS = 2
CONFIG = {
    "image_size": list(IMAGE_SIZE), "embedding_dim": EMBEDDING_DIM,
    "seed": SEED, "train_w": train_w, "val_w": val_w, "test_w": test_w,
    "preprocessing": "otsu",
    "margin": MARGIN, "epochs": EPOCHS, "freeze_epochs": FREEZE_EPOCHS, "batch_size": BATCH_SIZE,
}

In [ ]:
from torchvision import models


def build_pretrained(name, embedding_dim, pretrained=False):
    builders = {
        "resnet18": (models.resnet18, models.ResNet18_Weights.DEFAULT),
        "mobilenet_v3_small": (models.mobilenet_v3_small, models.MobileNet_V3_Small_Weights.DEFAULT),
        "efficientnet_b0": (models.efficientnet_b0, models.EfficientNet_B0_Weights.DEFAULT),
        "densenet121": (models.densenet121, models.DenseNet121_Weights.DEFAULT),
    }
    builder, weights = builders[name]
    backbone = builder(weights=weights if pretrained else None)
    if name == "resnet18":
        features = backbone.fc.in_features
        backbone.fc = nn.Identity()
    elif name == "densenet121":
        features = backbone.classifier.in_features
        backbone.classifier = nn.Identity()
    else:
        layer = backbone.classifier[0] if name == "mobilenet_v3_small" else backbone.classifier[-1]
        features = layer.in_features
        backbone.classifier = nn.Identity()
    model = nn.Sequential()
    model.add_module("backbone", backbone)
    model.add_module("head", nn.Sequential(
        nn.Linear(features, 256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, embedding_dim),
    ))
    return model

In [ ]:
if not 0 <= FREEZE_EPOCHS < EPOCHS:
    raise ValueError("EPOCHS must be positive and greater than FREEZE_EPOCHS.")
val_triplets = sample_triplets(writers, val_w, 32, SEED + 1)

for name in MODEL_NAMES:
    set_seed(SEED)
    path = MODEL_DIR / f"{name}_embedding.pt"
    loaded = path.is_file() and not FORCE_RETRAIN
    model = build_pretrained(name, EMBEDDING_DIM, pretrained=not loaded).to(DEVICE)
    if loaded:
        checkpoint = torch.load(path, map_location=DEVICE, weights_only=True)
        for key in ("image_size", "embedding_dim", "train_w", "val_w", "test_w", "preprocessing"):
            if checkpoint["config"].get(key) != CONFIG[key]:
                raise ValueError("Saved settings differ. Otsu preprocessing requires retraining. Set FORCE_RETRAIN = True.")
        model.load_state_dict(checkpoint["state_dict"])
        history = checkpoint["history"]
        print(name, "loaded. Set FORCE_RETRAIN = True to train again.")
    else:
        optimizer = torch.optim.Adam([
            {"params": model.backbone.parameters(), "lr": 1e-5},
            {"params": model.head.parameters(), "lr": 1e-4},
        ])
        best_loss, waiting = float("inf"), 0
        best_state = None
        history = {"train": [], "validation": []}
        for epoch in range(EPOCHS):
            frozen = epoch < FREEZE_EPOCHS
            for parameter in model.backbone.parameters():
                parameter.requires_grad = not frozen
            triplets = sample_triplets(writers, train_w, 64, SEED + 10 + epoch)
            train_loss = run_epoch(model, name, triplets, CONFIG, DEVICE, optimizer, frozen)
            val_loss = run_epoch(model, name, val_triplets, CONFIG, DEVICE)
            history["train"].append(train_loss)
            history["validation"].append(val_loss)
            print(f"{name} | Epoch {epoch + 1} | Train: {train_loss:.4f} | Validation: {val_loss:.4f}")
            if epoch == FREEZE_EPOCHS:
                best_loss, waiting = float("inf"), 0
            if val_loss < best_loss:
                best_loss = val_loss
                best_state = copy.deepcopy(model.state_dict())
                waiting = 0
            else:
                waiting += 1
            if epoch >= FREEZE_EPOCHS and waiting >= 3:
                break
        if best_state is None:
            raise ValueError("Training did not produce a finite validation loss.")
        model.load_state_dict(best_state)
        torch.save({"name": name, "state_dict": best_state, "config": CONFIG, "history": history}, path)
        print("Saved:", path)
    model.eval()
    plot_history(history, name)
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()